# Canonical AME(4,3) exact-optimized Bell baseline

This notebook materializes the canonical direct-basis AME(4,3) state using its exact optimized graph-state preparation. Run All always executes local Aer; IQM Garnet and PiastQ remain opt-in.

In [ ]:
import errno
import hashlib
import json
import os
import stat
import sys
import threading
import time
from contextlib import contextmanager
from pathlib import Path
from uuid import uuid4

import numpy as np
import qiskit.qpy as qpy
from qiskit.quantum_info import Operator, Statevector


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'qudits_on_qubits').is_dir():
            return candidate
    raise RuntimeError('Cannot find repository root containing pyproject.toml and src/qudits_on_qubits.')

def _checked_iqm_path(path, expected, description):
    path = Path(path)
    try:
        metadata = path.lstat()
    except FileNotFoundError:
        return None
    except OSError as error:
        raise RuntimeError(f'Cannot inspect {description}: {path}') from error
    reparse_attribute = getattr(stat, 'FILE_ATTRIBUTE_REPARSE_POINT', 0x400)
    if stat.S_ISLNK(metadata.st_mode) or bool(getattr(metadata, 'st_file_attributes', 0) & reparse_attribute):
        raise RuntimeError(f'Refusing symlink or reparse {description}: {path}')
    if expected == 'file' and not stat.S_ISREG(metadata.st_mode):
        raise RuntimeError(f'Malformed Git metadata or IQM .env candidate: expected file at {path}')
    if expected == 'dir' and not stat.S_ISDIR(metadata.st_mode):
        raise RuntimeError(f'Malformed Git metadata: expected directory at {path}')
    return path.resolve()


def resolve_iqm_env_path(repo_root):
    repo_root = Path(repo_root)
    git_metadata = repo_root / '.git'
    metadata = _checked_iqm_path(git_metadata, 'any', 'Git metadata')
    if metadata is None:
        raise RuntimeError(f'Missing Git metadata; cannot resolve IQM .env for {repo_root}')
    owning_repo = repo_root.resolve()
    if stat.S_ISREG(git_metadata.lstat().st_mode):
        try:
            gitdir_line = metadata.read_text(encoding='utf-8').strip()
        except OSError as error:
            raise RuntimeError(f'Malformed Git metadata: cannot read {git_metadata}') from error
        if not gitdir_line.lower().startswith('gitdir:'):
            raise RuntimeError(f'Malformed Git metadata: expected gitdir in {git_metadata}')
        gitdir_value = gitdir_line[7:].strip()
        if not gitdir_value:
            raise RuntimeError(f'Malformed Git metadata: empty gitdir in {git_metadata}')
        gitdir = Path(gitdir_value)
        if not gitdir.is_absolute():
            gitdir = metadata.parent / gitdir
        gitdir = _checked_iqm_path(gitdir, 'dir', 'Git worktree metadata')
        if gitdir is None:
            raise RuntimeError(f'Malformed Git metadata: missing worktree directory for {git_metadata}')
        commondir = _checked_iqm_path(gitdir / 'commondir', 'file', 'Git commondir metadata')
        if commondir is None:
            raise RuntimeError(f'Malformed Git metadata: missing commondir in {gitdir}')
        try:
            commondir_value = commondir.read_text(encoding='utf-8').strip()
        except OSError as error:
            raise RuntimeError(f'Malformed Git metadata: cannot read {commondir}') from error
        if not commondir_value:
            raise RuntimeError(f'Malformed Git metadata: empty commondir in {commondir}')
        common_git_dir = Path(commondir_value)
        if not common_git_dir.is_absolute():
            common_git_dir = gitdir / common_git_dir
        common_git_dir = _checked_iqm_path(common_git_dir, 'dir', 'Git common directory')
        if common_git_dir is None:
            raise RuntimeError(f'Malformed Git metadata: missing common directory for {gitdir}')
        owning_repo = _checked_iqm_path(common_git_dir.parent, 'dir', 'owning repository')
    candidates = [repo_root / '.env']
    if owning_repo != repo_root.resolve():
        candidates.append(owning_repo / '.env')
    for candidate in candidates:
        checked = _checked_iqm_path(candidate, 'file', 'IQM .env candidate')
        if checked is not None:
            return checked
    raise RuntimeError(f'Cannot find IQM .env file for checkout or owning repository: {repo_root}')


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qudits_on_qubits.reference_experiments import get_encoding, get_reference_experiment
from qudits_on_qubits.benchmarks.direct_basis.circuits import build_exact_optimized_direct_basis_graph_state_circuit
from qudits_on_qubits.experiments import (
    AerIdeal, BootstrapConfig, ExperimentSpec, IQMHardware, MitigationConfig,
    PathBasis, PiastQHardware, TranspilationConfig, run_experiment,
)


In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


class ExactOptimizedBasisFormatError(RuntimeError):
    pass


def is_symlink_or_reparse(path):
    try:
        metadata = Path(path).lstat()
    except OSError as error:
        raise RuntimeError(f'unable to inspect exact-optimized basis path: {path}') from error
    reparse_attribute = getattr(stat, 'FILE_ATTRIBUTE_REPARSE_POINT', 0x400)
    return stat.S_ISLNK(metadata.st_mode) or bool(getattr(metadata, 'st_file_attributes', 0) & reparse_attribute)


_PREPARE_LOCK_GUARD = threading.Lock()
_PREPARE_LOCKS = {}


def prepare_lock(repo_root):
    lock_key = str(Path(repo_root).resolve())
    with _PREPARE_LOCK_GUARD:
        return _PREPARE_LOCKS.setdefault(lock_key, threading.RLock())


def assert_safe_path_components(repo_root, path):
    repo_root = Path(repo_root)
    path = Path(path)
    try:
        relative_parts = path.relative_to(repo_root).parts
        resolved_root = repo_root.resolve(strict=True)
    except (OSError, ValueError) as error:
        raise RuntimeError(f'exact-optimized basis path is outside an available repository root: {path}') from error
    current = repo_root
    for part in ('', *relative_parts):
        if part:
            current = current / part
        try:
            current.lstat()
        except FileNotFoundError:
            continue
        except OSError as error:
            raise RuntimeError(f'unable to inspect exact-optimized basis path: {current}') from error
        if is_symlink_or_reparse(current):
            raise RuntimeError('exact-optimized basis path must not contain a symlink or reparse point')
        try:
            resolved_current = current.resolve(strict=True)
        except OSError as error:
            raise RuntimeError(f'unable to resolve exact-optimized basis path: {current}') from error
        if not resolved_current.is_relative_to(resolved_root):
            raise RuntimeError('exact-optimized basis path escapes the repository root')


def ensure_safe_directory(repo_root, directory):
    repo_root = Path(repo_root)
    directory = Path(directory)
    assert_safe_path_components(repo_root, directory)
    current = repo_root
    for part in directory.relative_to(repo_root).parts:
        current = current / part
        if not current.exists():
            try:
                current.mkdir()
            except FileExistsError:
                pass
        assert_safe_path_components(repo_root, current)
        if not current.is_dir():
            raise RuntimeError(f'exact-optimized basis path component must be a directory: {current}')


@contextmanager
def filesystem_prepare_lock(repo_root, parent):
    repo_root = Path(repo_root)
    parent = Path(parent)
    ensure_safe_directory(repo_root, parent)
    lock_path = parent / '.canonical_ez_exact_optimized.lock'
    assert_safe_path_components(repo_root, lock_path)
    with lock_path.open('a+b') as handle:
        assert_safe_path_components(repo_root, lock_path)
        handle.seek(0, 2)
        if handle.tell() == 0:
            handle.write(b'0')
            handle.flush()
        handle.seek(0)
        if os.name == 'nt':
            import msvcrt
            while True:
                try:
                    msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
                    break
                except OSError as error:
                    if error.errno not in {errno.EACCES, errno.EAGAIN}:
                        raise
                    time.sleep(0.05)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX)
        try:
            yield
        finally:
            handle.seek(0)
            if os.name == 'nt':
                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)


def remove_exact_optimized_bundle(repo_root, directory):
    directory = Path(directory)
    assert_safe_path_components(repo_root, directory)
    expected_files = {'graph_state_direct_basis.qpy', 'E.npy', 'metadata.json'}
    if {path.name for path in directory.iterdir()} != expected_files:
        raise RuntimeError(f'exact-optimized recovery directory has unexpected contents: {directory}')
    for path in directory.iterdir():
        if is_symlink_or_reparse(path):
            raise RuntimeError('exact-optimized recovery directory must not contain a symlink or reparse point')
        path.unlink()
    directory.rmdir()


def recover_interrupted_backup(repo_root, parent, directory):
    assert_safe_path_components(repo_root, parent)
    prefix = f'.{Path(directory).name}.legacy-'
    backups = [path for path in parent.iterdir() if path.name.startswith(prefix)]
    for backup in backups:
        assert_safe_path_components(repo_root, backup)
    if not backups:
        return
    if not Path(directory).exists():
        if len(backups) != 1:
            raise RuntimeError('exact-optimized basis recovery found multiple interrupted backups')
        backups[0].rename(directory)
        assert_safe_path_components(repo_root, directory)
        return
    for backup in backups:
        remove_exact_optimized_bundle(repo_root, backup)


def load_single_circuit(path):
    try:
        with Path(path).open('rb') as handle:
            circuits = qpy.load(handle)
    except Exception as error:
        raise RuntimeError(f'unable to load exact-optimized basis QPY: {path}') from error
    if len(circuits) != 1:
        raise RuntimeError(f'exact-optimized basis QPY must contain exactly one circuit, found {len(circuits)}')
    return circuits[0]


def validate_exact_optimized_basis(directory, expected_encoding, expected_circuit):
    directory = Path(directory)
    required_files = {'graph_state_direct_basis.qpy', 'E.npy', 'metadata.json'}
    try:
        actual_files = {path.name for path in directory.iterdir()}
    except OSError as error:
        raise RuntimeError(f'exact-optimized basis directory is unavailable: {directory}') from error
    if actual_files != required_files:
        raise RuntimeError(f'exact-optimized basis files are invalid: {sorted(actual_files)}')

    encoding_path = directory / 'E.npy'
    try:
        encoding = np.load(encoding_path, allow_pickle=False)
    except Exception as error:
        raise RuntimeError(f'exact-optimized basis encoding is invalid: {encoding_path}') from error
    if encoding.shape != (4, 3):
        raise RuntimeError('exact-optimized basis encoding must have shape (4, 3)')
    try:
        is_finite = np.isfinite(encoding).all()
    except TypeError as error:
        raise RuntimeError('exact-optimized basis encoding must be numeric and finite') from error
    if not is_finite:
        raise RuntimeError('exact-optimized basis encoding must be finite')
    if not np.allclose(encoding.conj().T @ encoding, np.eye(3), atol=1e-12, rtol=0):
        raise RuntimeError('exact-optimized basis encoding must be an isometry')
    if not np.array_equal(encoding, expected_encoding):
        raise RuntimeError('exact-optimized basis encoding does not match canonical_ez')

    circuit_path = directory / 'graph_state_direct_basis.qpy'
    circuit = load_single_circuit(circuit_path)
    if circuit.num_qubits != 8 or circuit.num_clbits != 0:
        raise RuntimeError('exact-optimized basis QPY must contain one unmeasured eight-qubit circuit')
    for instruction in circuit.data:
        operation = instruction.operation
        if operation.name in {'measure', 'reset'} or getattr(operation, 'condition', None) is not None or getattr(operation, 'blocks', ()):
            raise RuntimeError('exact-optimized basis QPY contains unsupported instructions')
    try:
        same_state = Statevector.from_instruction(circuit).equiv(Statevector.from_instruction(expected_circuit))
    except Exception as error:
        raise RuntimeError('exact-optimized basis QPY cannot be validated as state preparation') from error
    if not same_state:
        raise RuntimeError('exact-optimized basis QPY does not match the exact optimized graph state')

    metadata_path = directory / 'metadata.json'
    try:
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    except Exception as error:
        raise RuntimeError(f'exact-optimized basis metadata is invalid: {metadata_path}') from error
    expected_metadata = {
        'schema': 'qoq-reference-basis-v1', 'state': 'ame43', 'encoding_id': 'canonical_ez',
        'num_qubits': 8, 'encoding_shape': [4, 3],
        'files': {'graph_state_direct_basis.qpy': {'sha256': sha256_file(circuit_path)}, 'E.npy': {'sha256': sha256_file(encoding_path)}},
    }
    if metadata != expected_metadata:
        raise RuntimeError('exact-optimized basis metadata does not match the validated bundle')
    if len(circuit.data) != len(expected_circuit.data):
        raise ExactOptimizedBasisFormatError('exact-optimized basis QPY instruction count is stale')
    for actual, expected in zip(circuit.data, expected_circuit.data):
        actual_qargs = tuple(circuit.find_bit(qubit).index for qubit in actual.qubits)
        expected_qargs = tuple(expected_circuit.find_bit(qubit).index for qubit in expected.qubits)
        actual_cargs = tuple(circuit.find_bit(clbit).index for clbit in actual.clbits)
        expected_cargs = tuple(expected_circuit.find_bit(clbit).index for clbit in expected.clbits)
        if type(actual.operation) is not type(expected.operation) or actual_qargs != expected_qargs or actual_cargs != expected_cargs:
            raise ExactOptimizedBasisFormatError('exact-optimized basis QPY instruction structure is stale')
        try:
            same_operation = np.allclose(Operator(actual.operation).data, Operator(expected.operation).data, atol=1e-12, rtol=0)
        except Exception as error:
            raise ExactOptimizedBasisFormatError('exact-optimized basis QPY instruction cannot be structurally validated') from error
        if not same_operation:
            raise ExactOptimizedBasisFormatError('exact-optimized basis QPY instruction parameters are stale')


def _prepare_exact_optimized_basis_locked(repo_root):
    repo_root = Path(repo_root)
    expected_encoding = get_encoding('canonical_ez').as_array()
    expected_circuit = build_exact_optimized_direct_basis_graph_state_circuit('ame43', expected_encoding)
    parent = repo_root / 'experiment_inputs' / 'reference_bases' / 'ame43'
    directory = parent / 'canonical_ez_exact_optimized'
    ensure_safe_directory(repo_root, parent)
    recover_interrupted_backup(repo_root, parent, directory)
    assert_safe_path_components(repo_root, directory)
    rebuild_legacy = False
    if directory.exists():
        if is_symlink_or_reparse(directory):
            raise RuntimeError('exact-optimized basis directory must not be a symlink or reparse point')
        try:
            validate_exact_optimized_basis(directory, expected_encoding, expected_circuit)
        except ExactOptimizedBasisFormatError:
            rebuild_legacy = True
        else:
            return directory

    staging_directory = parent / f'.canonical_ez_exact_optimized.tmp-{uuid4().hex}'
    assert_safe_path_components(repo_root, parent)
    staging_directory.mkdir()
    assert_safe_path_components(repo_root, staging_directory)
    staging_files = tuple(staging_directory / name for name in ('graph_state_direct_basis.qpy', 'E.npy', 'metadata.json'))

    def cleanup_staging():
        assert_safe_path_components(repo_root, staging_directory)
        for staging_file in staging_files:
            if staging_file.exists():
                if is_symlink_or_reparse(staging_file):
                    raise RuntimeError('exact-optimized staging file must not be a symlink or reparse point')
                staging_file.unlink()
        if staging_directory.exists():
            staging_directory.rmdir()

    try:
        qpy_path, encoding_path, metadata_path = staging_files
        with qpy_path.open('wb') as handle:
            qpy.dump(expected_circuit, handle)
        with encoding_path.open('wb') as handle:
            np.save(handle, expected_encoding, allow_pickle=False)
        metadata = {
            'schema': 'qoq-reference-basis-v1', 'state': 'ame43', 'encoding_id': 'canonical_ez',
            'num_qubits': 8, 'encoding_shape': [4, 3],
            'files': {'graph_state_direct_basis.qpy': {'sha256': sha256_file(qpy_path)}, 'E.npy': {'sha256': sha256_file(encoding_path)}},
        }
        metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
        validate_exact_optimized_basis(staging_directory, expected_encoding, expected_circuit)
        backup_directory = None
        if rebuild_legacy:
            backup_directory = parent / f'.canonical_ez_exact_optimized.legacy-{uuid4().hex}'
            try:
                assert_safe_path_components(repo_root, directory)
                directory.rename(backup_directory)
            except FileNotFoundError:
                backup_directory = None
            else:
                assert_safe_path_components(repo_root, backup_directory)
        try:
            assert_safe_path_components(repo_root, parent)
            staging_directory.rename(directory)
        except FileExistsError:
            cleanup_staging()
            validate_exact_optimized_basis(directory, expected_encoding, expected_circuit)
        except BaseException:
            if backup_directory is not None and not directory.exists():
                assert_safe_path_components(repo_root, backup_directory)
                backup_directory.rename(directory)
            raise
        if backup_directory is not None:
            remove_exact_optimized_bundle(repo_root, backup_directory)
        return directory
    finally:
        cleanup_staging()


def prepare_exact_optimized_basis(repo_root):
    repo_root = Path(repo_root)
    with prepare_lock(repo_root):
        parent = repo_root / 'experiment_inputs' / 'reference_bases' / 'ame43'
        with filesystem_prepare_lock(repo_root, parent):
            return _prepare_exact_optimized_basis_locked(repo_root)


In [ ]:
CANONICAL_BASIS_DIRECTORY = prepare_exact_optimized_basis(REPO_ROOT)
CANONICAL_BASIS_DIRECTORY


In [ ]:
SHOTS = 100
UNCERTAINTY = BootstrapConfig(samples=2_000, seed=7)
HARDWARE_MITIGATION = MitigationConfig(readout=True, zne=True, zne_factors=(1, 3, 5))
TRANSPILATION = TranspilationConfig(optimization_level=3, seed_transpiler=13)
REFERENCE = get_reference_experiment('ame43')
RESULTS = {}


## Aer exact-optimized baseline


In [ ]:
AER_SPEC = ExperimentSpec(
    state='ame43', basis=PathBasis(CANONICAL_BASIS_DIRECTORY), backend=AerIdeal(seed_simulator=11),
    shots=SHOTS, uncertainty=UNCERTAINTY, transpilation=TRANSPILATION,
    tags={'baseline': 'canonical_ez', 'preparation': 'exact_optimized', 'backend': 'aer_ideal'},
)
AER_RESULT = run_experiment(AER_SPEC, repo_root=REPO_ROOT)
RESULTS['aer_ideal'] = AER_RESULT


## IQM Garnet exact-optimized baseline


In [ ]:
RUN_IQM = False

if RUN_IQM:
    IQM_ENV_PATH = resolve_iqm_env_path(REPO_ROOT)
    IQM_SPEC = ExperimentSpec(
        state='ame43', basis=PathBasis(CANONICAL_BASIS_DIRECTORY), backend=IQMHardware(device='garnet', use_metrics=True, env_path=IQM_ENV_PATH),
        shots=SHOTS, mitigation=HARDWARE_MITIGATION, uncertainty=UNCERTAINTY, transpilation=TRANSPILATION,
        tags={'baseline': 'canonical_ez', 'preparation': 'exact_optimized', 'backend': 'iqm_garnet'},
    )
    RESULTS['iqm_garnet'] = run_experiment(IQM_SPEC, repo_root=REPO_ROOT)
else:
    print('IQM Garnet skipped; set RUN_IQM = True to submit.')


## PiastQ exact-optimized baseline


In [ ]:
RUN_PIASTQ = False

if RUN_PIASTQ:
    PIASTQ_SPEC = ExperimentSpec(
        state='ame43', basis=PathBasis(CANONICAL_BASIS_DIRECTORY), backend=PiastQHardware(mode='managed', owner='notebook'),
        shots=SHOTS, mitigation=HARDWARE_MITIGATION, uncertainty=UNCERTAINTY, transpilation=TRANSPILATION,
        tags={'baseline': 'canonical_ez', 'preparation': 'exact_optimized', 'backend': 'piastq'},
    )
    RESULTS['piastq'] = run_experiment(PIASTQ_SPEC, repo_root=REPO_ROOT)
else:
    print('PiastQ skipped; set RUN_PIASTQ = True to submit.')


In [ ]:
def summarize_results(results, reference):
    return [
        {
            'backend': backend,
            'status': results[backend].status.value if backend in results else missing_status,
            'result': None if backend not in results else results[backend].values,
            'classical_bound': reference.bell_functional.classical_bound,
            'ideal_bell_value': reference.expected.ideal_bell_value,
        }
        for backend, missing_status in (('aer_ideal', 'not_run'), ('iqm_garnet', 'skipped'), ('piastq', 'skipped'))
    ]


SUMMARY = summarize_results(RESULTS, REFERENCE)
SUMMARY
